In [29]:
!pip install -q groq

from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv("key.env")

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

client = Groq(api_key = GROQ_API_KEY)

In [38]:
#helper function to talk to LLM
def query_llm(model_name : str, chat_history : list, temperature = 0.2, n = 1):
    try:
        messages = []
        for msg in chat_history:
            #Makes sure the role is one of these three valid values
            if msg["role"] in ["system", "user", "assistant"]:
                #Creates new dictionary of filtered values
                messages.append({
                    "role" : msg["role"],
                    "content" : msg["content"]
                })

        #Has the chat model generate a response
        completion = client.chat.completions.create(
            messages = messages, #conversation history that was prepared
            model = model_name, #string for which model to use
            temperature = temperature, #randomness of the model
            n=n
        )

        #if there is a response, return the first one unless 
        #number of responses higher than one
        if completion.choices and len(completion.choices) > 0:
            if n == 1:
                return completion.choices[0].message.content
            else:
                return [c.message.content for c in completion.choices]
            
        return "No response generated"

    #If there is an error, return this message instead of just crashing
    except Exception as e:
        return f"Error querying the model: {e}"

In [39]:
prompt = "What LLM are you?"

#Calling function with values for model and chat history
response = query_llm(model_name = "gemma2-9b-it", chat_history = [{"role" : "user", 
                                                                   "content" : prompt}])
print(response)

I am Gemma, an open-weights AI assistant developed by the Gemma team at Google DeepMind.



In [32]:
#function to determine the llm is working and if not where something is going wrong
def test_groq_connection(model_name: str = "gemma2-9b-it"):
    try:
        # Initialize client
        if not GROQ_API_KEY:
            print("Error: No API key found in userdata")
            return False

        print(f"API key length: {len(GROQ_API_KEY)}")

        client = Groq(api_key=GROQ_API_KEY)
        print("HI!")

        completion = client.chat.completions.create(
            messages=[{"role": "user", "content": "Hello"}],
            model=model_name
        )

        print("Connection successful!")
        return True

    except Exception as e:
        print(f"Detailed error information:")
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {str(e)}")
        return False

print("\nNow testing connection...")
test_groq_connection()


Now testing connection...
API key length: 56
HI!
Connection successful!


True

In [74]:
#Prompt templates
ZERO_SHOT_PROMPT = """
System: You are an assistant recommending video games based on user preferences.
"""


FEW_SHOT_PROMPT = """
System: You are an assistant recommending video games based on user preferences
User: Recommend me a movie based on the following examples:

Example 1:
    Preferences: Platforming, Action, 2D, recently enjoyed "Hollow Knight"
    Recommendation: Ori and the Blind Forest

Example 2:
    Preferences: Multiplayer, role-playing game, recently enjoyed "World of Warcraft"
    Recommendation: Runescape

"""

CHAIN_OF_THOUGHT_PROMPT = """
System: You are an assistant who thinks step by step to provide accurate video game recommendations.

Answer: Let's think step by step.
Step 1: Identify the user's preferenes for the video game (genre, artstyle, pc/console, gameplay, single/multiplayer)
Step 2: Search for video games that fit the users preferences.
Step 3: Filter the video games are more popular as reasoning for the user to also enjoy them.
Step 4: Compare the remaining video games to one the user likes (if specified) and select the one that is most similar.
Step 5: Provide the recommendation with a brief explanation.
"""


In [ ]:
def chatbot():
    while True:
        print("Welcome to the Video Game Recommendation Chatbot!")

        print("\nAvailable models:")
        print("1. gemma2-9b-it")
        print("2. llama-3.1-8b-instant")
        print("3. openai/gpt-oss-120b")
        model_choice = input("Enter model number (1, 2, or 3: )").strip()

        model_map = {
        "1" : "gemma2-9b-it",
        "2" : "llama-3.1-8b-instant",
        "3" : "openai/gpt-oss-120b"
        }
        
        if model_choice not in model_map:
            print("Invalid model choice, restart chatbot.")
            continue

        model_name = model_map[model_choice]
        chat_history = []

        print("\nChoose a prompting technique:")
        print("1. Zero-Shot")
        print("2. Few-Shot")
        print("3. Chain-of-Thought")
        choice = input("Enter your choice (1, 2, or 3) or 'exit' to quit: ").strip()

        if choice.lower() == 'exit':
            print("Goodbye!")
            break

        if choice not in ['1', '2', '3', '4']:
            print("Invalid choice!")
            continue

        if choice == "1":
            system_message = ZERO_SHOT_PROMPT
        elif choice == "2":
            system_message = FEW_SHOT_PROMPT
        else:
            system_message = CHAIN_OF_THOUGHT_PROMPT

        chat_history.append({"role": "system", "content": system_message.strip()})
    
        response = query_llm(model_name, chat_history)
        print(f"\nAssistant: {response}")

        while True:
            print("\nOptions:")
            print("1. Continue conversation")
            print("2. Start new conversation")
            print("3. Perform model graded evaluation on previous response")
            print("4. Exit")

            option = input("Choose an option (1-4): ").strip()

            if option == '1':
                user_message = input("\nYou: ").strip()

                if user_message.lower() == 'exit':
                    print("Goodbye!")
                    break

                chat_history.append({"role": "user", "content": user_message})
                response = query_llm(model_name, chat_history)
                chat_history.append({"role": "assistant", "content": response})
                print(f"\nAssistant: {response}")

            elif option =='2':
                break

            elif option == '3':
                if len(chat_history) >= 2 and chat_history[-2]["role"] == "user":
                    user_message = chat_history[-2]["content"]
                else:
                    user_message = "(no user prompt provided)"
                response = chat_history[-1]["content"]
                MGE_prompt = f"""Given the system response: {response}, input prompt: {user_message}, model name: {model_name}
                Evaluate of how well the system response answers the input prompt...."""

                eval_result = query_llm(model_name, [{"role": "user", "content": MGE_prompt}])
                print(eval_result)

            elif option == '4':
                print("Goodbye!")
                return

            else:
                print("Invalid option!")

if __name__ == "__main__":
    chatbot()
            
        

Welcome to the Video Game Recommendation Chatbot!

Available models:
1. gemma2-9b-it
2. llama-3.1-8b-instant
3. openai/gpt-oss-120b


Enter model number (1, 2, or 3: ) 2



Choose a prompting technique:
1. Zero-Shot
2. Few-Shot
3. Chain-of-Thought


Enter your choice (1, 2, or 3) or 'exit' to quit:  3



Assistant: I'm ready to provide a video game recommendation. To get started, I'll need some information from you. Please answer the following questions:

1. What genre of video games are you interested in (e.g. action, adventure, role-playing, strategy, sports, etc.)?
2. What type of art style do you prefer (e.g. realistic, cartoonish, futuristic, historical, etc.)?
3. Do you play on PC or console (e.g. PlayStation, Xbox, Nintendo Switch, etc.)?
4. What type of gameplay do you enjoy (e.g. exploration, puzzle-solving, combat, building, etc.)?
5. Do you prefer playing single-player or multi-player games?
6. Is there a specific video game that you like and would like me to compare the recommendation to?

Please answer these questions, and I'll start the process of finding a video game recommendation for you.

Options:
1. Continue conversation
2. Start new conversation
3. Perform model graded evaluation on previous response
4. Exit


Choose an option (1-4):  1

You:  cute, story, dont care about art, PC, building, multiplayer, genshin impact



Assistant: Let's think step by step to find a video game recommendation for you.

**Step 1: Identify the user's preferences**
- Genre: Cute and story-driven games
- Platform: PC
- Gameplay: Building and multiplayer
- Similar to: Genshin Impact (open-world, action RPG)

**Step 2: Search for video games that fit the user's preferences**
Based on your preferences, here are some video games that might interest you:
- Stardew Valley (farming simulation, building, multiplayer)
- Animal Crossing: New Horizons (life simulation, building, multiplayer)
- Terraria (action-adventure, building, multiplayer)
- Little Dragons Café (story-driven, farming simulation, building)
- My Time at Portia (life simulation, building, multiplayer)

**Step 3: Filter the video games based on popularity**
Based on user reviews and ratings, here are the top 3 video games from the previous list:
- Stardew Valley (average rating: 9/10)
- Animal Crossing: New Horizons (average rating: 9/10)
- Terraria (average rating: 